# Holographic Axicon Beam Study

Stage C notebook for the SLM-encoded holographic axicon. Runs general and limits regimes through the corrected air beam-to-surface study.

In [1]:
from dataclasses import replace
from pathlib import Path
import pandas as pd
import bessel_twin_core as bt
from vbb_study import vbb_regime, vbb_train_viz
base = replace(bt.default_config('fast'), generation_method='holographic')
project_dir = Path.cwd() if Path.cwd().name == 'Publication_Study' else Path.cwd()/'Publication_Study'
out_fig = project_dir/'outputs/figures/stage_c'
out_csv = project_dir/'outputs/csv/stage_c'
out_fig.mkdir(parents=True, exist_ok=True); out_csv.mkdir(parents=True, exist_ok=True)

In [2]:
rows = []
for regime in ('general', 'limits'):
    cfg = vbb_regime.config_for_regime(base, regime)
    ideal = bt.run_case(cfg, preset='fast', path='ideal', case_id=f'{regime}_holographic_ideal')
    lab = bt.run_case(cfg, preset='fast', path='realistic', case_id=f'{regime}_holographic_lab')
    for label, result in [('ideal', ideal), ('lab', lab)]:
        m = result['metrics']
        rows.append({'regime': regime, 'path': label, 'zone_um': m['bessel_zone_um'], 'feature_um': m['feature_diameter_um'], 'peak_fluence_J_cm2': m['peak_fluence_J_cm2'], 'contrast': m['side_to_core_peak_ratio'], 'valid': result['validity_report']['valid']})
summary = pd.DataFrame(rows)
summary.to_csv(out_csv/'NB_holographic_summary.csv', index=False)
summary

,regime,path,zone_um,feature_um,peak_fluence_J_cm2,contrast,valid
0,general,ideal,116.406260,5.277632,2.674508,0.429279,True
1,general,lab,38.015567,3.270841,4.718381,0.376263,True
2,limits,ideal,114.468843,3.267105,3.149295,0.476238,True
3,limits,lab,249.745519,4.776797,0.605325,1.000000,True


In [3]:
delta = summary.pivot(index='regime', columns='path', values=['zone_um','feature_um','peak_fluence_J_cm2','contrast'])
delta

zone_um             feature_um           peak_fluence_J_cm2  \
path          ideal         lab      ideal       lab              ideal   
regime                                                                    
general  116.406260   38.015567   5.277632  3.270841           2.674508   
limits   114.468843  249.745519   3.267105  4.776797           3.149295   

                   contrast            
path          lab     ideal       lab  
regime                                 
general  4.718381  0.429279  0.376263  
limits   0.605325  0.476238  1.000000

In [4]:
vbb_train_viz.plot_train_visualiser(base, method='holographic', output_dir=out_fig)
vbb_train_viz.plot_sampling_qa(base, output_dir=out_fig)

WindowsPath('C:/PhD/Code/Publication_Study/outputs/figures/stage_c/stage_c_sampling_qa_limits.png')

## Stage C.5 carrier/filter fairness sweep

Bounded nominal ell=3 sweep of blaze period and filter radius. The chosen point maximises first-order selected fraction while keeping zero-order leakage below the stated threshold and avoiding a capped canonical zone.

In [5]:
# Stage C.5 carrier/filter fairness sweep
carrier_sweep = vbb_train_viz.holographic_carrier_filter_sweep(base)
carrier_sweep.to_csv(out_csv/'NB_holographic_carrier_filter_sweep.csv', index=False)
vbb_train_viz.plot_holographic_carrier_filter_tradeoff(carrier_sweep, base, output_dir=out_fig)
carrier_sweep.loc[carrier_sweep['is_optimum']].reset_index(drop=True)

,blaze_period_px,carrier_lpmm,configured_filter_radius_lpmm,effective_filter_radius_lpmm,recommended_filter_radius_lpmm,axicon_cone_radius_lpmm,first_order_selected_fraction,zero_order_leakage_total_fraction,zero_order_leakage_dc_fraction,bessel_zone_um,zone_capped,first_order_geometry_valid,phase_sampling_label,focal_sampling_label,validity_valid,surface_z_um,is_eligible_optimum,is_optimum
0,18,6.944444,6.0,6.0,6.0,5.02533,0.987891,0.000011,0.034511,35.211526,False,True,pass,pass,True,-55.562596,True,True


In [6]:
# Stage C.5 fair rerun: use the optimised holographic carrier/filter in the method comparison.
optimum = carrier_sweep.loc[carrier_sweep['is_optimum']].iloc[0]
optimised_base = replace(
    base,
    slm=replace(
        base.slm,
        blaze_period_px=int(optimum['blaze_period_px']),
        first_order_filter_radius_lpmm=float(optimum['configured_filter_radius_lpmm']),
    ),
)
fair_comparison = vbb_train_viz.method_comparison_table(optimised_base, regimes=('general',), methods=('holographic','physical'))
fair_comparison.to_csv(out_csv/'NB_holographic_optimised_method_comparison.csv', index=False)
fair_comparison

,regime,method,path,bessel_zone_um,feature_diameter_um,peak_fluence_J_cm2,side_to_core_peak_ratio,throughput_or_efficiency,realised_k_r_m_inv,slm2_residual_before_rad,slm2_residual_after_rad,validity_valid
0,general,holographic,realistic,35.211526,3.801189,5.275187,0.305576,0.987891,1.603333e+06,NaN,NaN,True
1,general,physical,ideal,113.937100,4.775000,14.381208,0.539002,0.950288,1.603333e+06,1.813817,0.007127,True
